# 11 — Training Loop — Solution

---

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from src.models.transformer import TransformerLM, TransformerConfig
from src.training.trainer import Trainer, TrainingConfig

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Part A — LM Loss

In [ ]:
def lm_loss(logits, targets, ignore_index=-100):
    vocab_size = logits.size(-1)
    # Flatten: (B*S, V) and (B*S,)
    return F.cross_entropy(
        logits.view(-1, vocab_size),
        targets.view(-1),
        ignore_index=ignore_index
    )


vocab, batch, seq = 50, 4, 16
logits_t  = torch.randn(batch, seq, vocab)
targets_t = torch.randint(0, vocab, (batch, seq))
loss = lm_loss(logits_t, targets_t)
print(f'LM Loss: {loss.item():.4f}   Expected ~log({vocab})={math.log(vocab):.4f}')

## Part B — LR Schedule

In [ ]:
def get_lr(step, warmup_steps, max_steps, peak_lr=3e-4, min_lr=1e-5):
    if step < warmup_steps:
        return peak_lr * step / warmup_steps
    # Cosine decay from peak_lr to min_lr
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))


max_steps, warmup = 1000, 100
lrs = [get_lr(s, warmup, max_steps) for s in range(max_steps)]
plt.figure(figsize=(8, 3))
plt.plot(lrs); plt.axvline(warmup, color='r', linestyle='--', label=f'Warmup @ step {warmup}')
plt.xlabel('Step'); plt.ylabel('Learning rate')
plt.title('Warmup + Cosine Decay'); plt.legend()
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## Part C — Training Loop

In [ ]:
cfg = TransformerConfig(
    vocab_size=64, d_model=64, n_heads=4, n_layers=2,
    d_ff=128, max_seq_len=32, dropout=0.1,
    ffn_type='standard', norm_type='layernorm', pos_encoding='sinusoidal',
)
model = TransformerLM(cfg).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)


def train_step(model, optimizer, batch, grad_clip=1.0, step=0,
               warmup_steps=50, max_steps=500, peak_lr=3e-4):
    model.train()
    batch = batch.to(device)
    x, target = batch[:, :-1], batch[:, 1:]

    # Forward
    logits = model(x)

    # Loss
    loss = lm_loss(logits, target)

    # Update learning rate
    lr = get_lr(step, warmup_steps, max_steps, peak_lr)
    for g in optimizer.param_groups:
        g['lr'] = lr

    # Backward + clip + step
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    return loss.item()


data   = torch.randint(0, cfg.vocab_size, (500, cfg.max_seq_len))
loader = DataLoader(TensorDataset(data), batch_size=32, shuffle=True)

losses, step = [], 0
for epoch in range(5):
    for (batch,) in loader:
        loss_val = train_step(model, optimizer, batch, step=step)
        losses.append(loss_val)
        step += 1
    print(f'Epoch {epoch+1}  loss={sum(losses[-len(loader):])/len(loader):.4f}')

plt.figure(figsize=(8, 3))
plt.plot(losses, alpha=0.6, color='steelblue')
plt.axhline(math.log(cfg.vocab_size), color='r', linestyle='--', label='Chance level')
plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('Training Loss'); plt.legend()
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()